# ML Model Training & Testing — Google Colab

**Teacher's workflow:** Train and evaluate the model in Google Colab, save `model.joblib`, then download that artifact and use it in VS Code for FastAPI and Docker.

Dataset: Wisconsin Diagnostic Breast Cancer. This notebook downloads the CSV from the public Kaggle dataset page's commonly used GitHub mirror so it can run in Colab without Kaggle API credentials. If your teacher requires the Kaggle download itself, upload the Kaggle CSV using the upload cell below.

> Educational demonstration only — not for medical diagnosis.

## 1. Install dependencies (Colab)

In [ ]:
!pip -q install scikit-learn pandas joblib

## 2. Get the dataset

Option A: load the public Wisconsin diagnostic CSV automatically.

Option B: upload the CSV you downloaded from Kaggle. If you upload one, set `USE_UPLOADED_KAGGLE_CSV = True` below.

In [ ]:
from pathlib import Path
import pandas as pd

USE_UPLOADED_KAGGLE_CSV = False

if USE_UPLOADED_KAGGLE_CSV:
    from google.colab import files
    uploaded = files.upload()
    csv_name = next(name for name in uploaded if name.lower().endswith(".csv"))
    df = pd.read_csv(csv_name)
else:
    url = "https://raw.githubusercontent.com/plotly/datasets/master/data.csv"
    df = pd.read_csv(url)

print("Dataset shape:", df.shape)
display(df.head())

## 3. Clean and normalize the dataset columns

The model expects the 30 numeric measurement features. Kaggle's familiar column names such as `radius_mean` are converted to the scikit-learn names such as `mean radius`.

In [ ]:
rename_map = {
    "radius_mean": "mean radius", "texture_mean": "mean texture",
    "perimeter_mean": "mean perimeter", "area_mean": "mean area",
    "smoothness_mean": "mean smoothness", "compactness_mean": "mean compactness",
    "concavity_mean": "mean concavity", "concave points_mean": "mean concave points",
    "symmetry_mean": "mean symmetry", "fractal_dimension_mean": "mean fractal dimension",
    "radius_se": "radius error", "texture_se": "texture error",
    "perimeter_se": "perimeter error", "area_se": "area error",
    "smoothness_se": "smoothness error", "compactness_se": "compactness error",
    "concavity_se": "concavity error", "concave points_se": "concave points error",
    "symmetry_se": "symmetry error", "fractal_dimension_se": "fractal dimension error",
    "radius_worst": "worst radius", "texture_worst": "worst texture",
    "perimeter_worst": "worst perimeter", "area_worst": "worst area",
    "smoothness_worst": "worst smoothness", "compactness_worst": "worst compactness",
    "concavity_worst": "worst concavity", "concave points_worst": "worst concave points",
    "symmetry_worst": "worst symmetry", "fractal_dimension_worst": "worst fractal dimension",
}
df = df.rename(columns=rename_map)
df = df.drop(columns=[c for c in ["id", "Unnamed: 32", "target"] if c in df.columns], errors="ignore")

if "diagnosis" not in df.columns:
    raise ValueError("Expected a 'diagnosis' target column with B/M values.")

df = df[["diagnosis"] + [c for c in df.columns if c != "diagnosis"]]
print("Rows and columns:", df.shape)
print(df["diagnosis"].value_counts())
display(df.head())

## 4. Train/test split and train model

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

feature_names = [c for c in df.columns if c != "diagnosis"]
X = df[feature_names].apply(pd.to_numeric, errors="coerce")
y = (df["diagnosis"].astype(str).str.upper() == "M").astype(int)  # 1=malignant, 0=benign

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=2000, random_state=42)),
])
model.fit(X_train, y_train)
print("Model training complete.")
print("Training rows:", len(X_train), "| Testing rows:", len(X_test))

## 5. Test and evaluate the trained model in Colab

In [ ]:
y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\\nClassification report:")
print(classification_report(y_test, y_pred, target_names=["benign", "malignant"]))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

## 6. Save the trained model as `model.joblib`

This file contains the fitted pipeline and feature metadata. We will transfer it to VS Code for API prediction.

In [ ]:
import joblib

artifact = {
    "model": model,
    "feature_names": feature_names,
    "target_labels": {"0": "benign", "1": "malignant"},
}
joblib.dump(artifact, "model.joblib")
print("Saved:", Path("model.joblib").resolve())

## 7. Download `model.joblib` to your computer

In [ ]:
from google.colab import files
files.download("model.joblib")

## 8. Optional: download a sample prediction payload

The VS Code API expects the 30 feature values as JSON. This creates a sample payload from the first test row, using API-friendly field names.

In [ ]:
api_names = [
    "mean_radius","mean_texture","mean_perimeter","mean_area","mean_smoothness","mean_compactness","mean_concavity","mean_concave_points","mean_symmetry","mean_fractal_dimension",
    "radius_error","texture_error","perimeter_error","area_error","smoothness_error","compactness_error","concavity_error","concave_points_error","symmetry_error","fractal_dimension_error",
    "worst_radius","worst_texture","worst_perimeter","worst_area","worst_smoothness","worst_compactness","worst_concavity","worst_concave_points","worst_symmetry","worst_fractal_dimension"
]
sample = X_test.iloc[0]
payload = {api_name: float(sample[col]) for api_name, col in zip(api_names, feature_names)}
with open("sample_payload.json", "w") as f:
    import json
    json.dump(payload, f, indent=2)
files.download("sample_payload.json")